In [ ]:
#!pip install xgboost lightgbm -q

In [7]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from dotenv import load_dotenv
import os
load_dotenv()
DF_PATH = os.getenv('DF_MODEL')
df = pd.read_csv(DF_PATH)
df.head()

,companycode,ano,periodo,ativos_circulantes,ativos_não_circulantes,total_de_ativos,passivos_circulantes,passivos_não_circulantes,total_do_passivo_e_patrimônico_líquido,patrimônio_líquido,...,fco_divida,fcf_divida,cobertura_juros,ebitda_divida,giro_ativos,ciclo_estoques,ciclo_recebiveis,ciclo_pagamentos,ciclo_estoques_dias,score_total
0,ADEL,2008,3_Meses,82082356.0,16000553.0,98082909.0,49511334.0,2498829.0,98082909.0,46072746.0,...,-4.509550,-4.664420,-18.401068,0.841152,0.242154,-2.010322,2.374165,-0.169180,-733.767538,2.2875
1,ADEL,2008,6_Meses,88600612.0,16099148.0,104699760.0,55983895.0,2490529.0,104699760.0,46225336.0,...,-1.489560,-1.540716,-10.595002,0.513515,0.468567,-0.939450,1.268414,-0.101706,-342.899268,2.4125
2,ADEL,2008,9_Meses,94009683.0,15551293.0,109560976.0,54753842.0,2571602.0,109560976.0,52235532.0,...,-1.103726,-1.141631,-6.868930,0.626646,0.687130,-0.628661,0.916802,-0.038461,-229.461288,2.6500
3,ADEL,2008,Anual,45582654.0,15170747.0,60753401.0,8454967.0,2891157.0,60753401.0,49407277.0,...,-13.849306,-14.324928,-4.650982,7.076576,1.231006,-0.729009,0.046323,-0.035645,-266.088413,3.0625
4,ADEL,2009,3_Meses,82082356.0,16000553.0,98082909.0,49511334.0,2498829.0,98082909.0,46072746.0,...,-4.509550,-4.664420,-18.401068,0.841152,0.242154,-2.010322,2.374165,-0.169180,-733.767538,2.2875


In [8]:
# Seus dados já preparados (mesma divisão temporal que você usou)
df_train = df[df['data'] < '2024-01-01']
df_test  = df[df['data'] >= '2024-01-01']

feat = [c for c in df.columns if c not in ['score_total', 'companycode', 'ano', 'periodo', 'data']]
X_train, y_train = df_train[feat], df_train['score_total']
X_test,  y_test  = df_test[feat],  df_test['score_total']

# Definir os 4 modelos — todos dentro de Pipeline para tratar nulos
modelos = {
    "Ridge (baseline)": Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
        ('model',   Ridge(alpha=1.0))
    ]),
    "Random Forest": Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
        ('model',   RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
    ]),
    "XGBoost": Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
        ('model',   XGBRegressor(n_estimators=200, learning_rate=0.05,
                                  max_depth=6, random_state=42, n_jobs=-1))
    ]),
    "LightGBM": Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
        ('model',   LGBMRegressor(n_estimators=200, learning_rate=0.05,
                                   max_depth=6, random_state=42, n_jobs=-1))
    ]),
}

# Treinar e avaliar todos
resultados = []

for nome, pipeline in modelos.items():
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)

    resultados.append({
        'Modelo':  nome,
        'R²':      round(r2_score(y_test, pred), 4),
        'MAE':     round(mean_absolute_error(y_test, pred), 4),
        'RMSE':    round(np.sqrt(mean_squared_error(y_test, pred)), 4),
        'MAPE (%)': round(np.mean(np.abs((y_test - pred) / (y_test + 1e-6))) * 100, 2),
    })

df_resultados = pd.DataFrame(resultados).sort_values('R²', ascending=False)
print(df_resultados.to_string(index=False))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001833 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11475
[LightGBM] [Info] Number of data points in the train set: 14460, number of used features: 45
[LightGBM] [Info] Start training from score 1.869334
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
          Modelo     R²    MAE   RMSE  MAPE (%)
        LightGBM 0.9889 0.0482 0.0660      2.90
         XGBoost 0.9864 0.0527 0.0730      3.22
   Random Forest 0.9614 0.0934 0.1230      6.09
Ridge (baseline

c:\Users\Lucas\Documents\usp\TCC-CD-USP\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
